In [ ]:
import sys
from pathlib import Path

from clients.db_client import DBClient
from clients.kraken_csv_client import KrakenCSVClient

# Initialize clients
db = DBClient()
kraken = KrakenCSVClient(data_path="../../data")

In [ ]:
# Import Kraken CSV data
csv_files = kraken.load_csv_files()
print(f"Found {len(csv_files)} CSV file(s)\n")

total_inserted = 0

for file_path in csv_files:
    ticker = kraken.get_ticker_from_filename(file_path)
    
    print(f"Processing {ticker}...")
    
    # Check if instrument exists in database
    instrument_id = db.get_instrument_id(ticker, exchange="kraken")
    
    if instrument_id:
        # Parse CSV and insert data
        df = kraken.parse_csv(file_path)
        df['instrument_id'] = instrument_id  # Add the ID
        rows = db.insert_market_data(instrument_id, df)
        total_inserted += rows
        print(f"Inserted rows")
    else:
        print(f"  ✗ Not found in instruments table")

print(f"Total rows inserted: {total_inserted}")

In [ ]:
df